In [0]:
%python
# Databricks notebook source
# MAGIC %md
# MAGIC # Module 2 -- Conversation Classifier (AI vs Rule-Based)
# MAGIC
# MAGIC Samples real conversations, classifies each with Claude (intent / sentiment /
# MAGIC frustration) and with a lightweight regex-based classifier, and compares the two.
# MAGIC
# MAGIC **Important framing:** this notebook reports an **agreement rate** between Claude
# MAGIC and the rule-based classifier -- not an "accuracy" number. Neither side has been
# MAGIC checked against human-labeled ground truth, so there's no basis to call one of them
# MAGIC "correct." A cell near the end can export a small CSV for you to hand-label yourself
# MAGIC if you want a real accuracy figure later -- that's a separate, optional step.
# MAGIC
# MAGIC **Cost control:** this samples a small number of conversations by default (see
# MAGIC `SAMPLE_SIZE` below), not the full dataset. `estimate_cost()` runs before any API
# MAGIC calls so you see the bill before it happens.

# COMMAND ----------

if "call_claude" not in globals() or "estimate_cost" not in globals():
    try:
        import anthropic
    except ModuleNotFoundError as e:
        raise ModuleNotFoundError(
            "anthropic is not installed in this session. Run 01_claude_api_setup.py first to install/configure it."
        ) from e

    import time
    import hashlib
    import json

    SECRET_SCOPE = "claude-api"
    SECRET_KEY = "anthropic-key"
    CLAUDE_MODEL = "claude-sonnet-5"

    CLAUDE_API_KEY = dbutils.secrets.get(scope=SECRET_SCOPE, key=SECRET_KEY)
    client = anthropic.Anthropic(api_key=CLAUDE_API_KEY)

    class RateLimiter:
        def __init__(self, min_interval_seconds: float = 1.0):
            self.min_interval = min_interval_seconds
            self._last_call = 0.0

        def wait(self):
            elapsed = time.time() - self._last_call
            if elapsed < self.min_interval:
                time.sleep(self.min_interval - elapsed)
            self._last_call = time.time()

    rate_limiter = RateLimiter(min_interval_seconds=1.0)
    _response_cache = {}

    def _call_with_retry(prompt: str, model: str = CLAUDE_MODEL, max_tokens: int = 1024,
                         max_retries: int = 3, **kwargs):
        last_error = None
        for attempt in range(max_retries):
            try:
                rate_limiter.wait()
                return client.messages.create(
                    model=model,
                    max_tokens=max_tokens,
                    messages=[{"role": "user", "content": prompt}],
                    **kwargs,
                )
            except anthropic.RateLimitError as e:
                wait_time = 2 ** attempt * 2
                print(f"Rate limited -- retrying in {wait_time}s (attempt {attempt + 1}/{max_retries})")
                time.sleep(wait_time)
                last_error = e
            except anthropic.APIStatusError as e:
                if e.status_code >= 500:
                    wait_time = 2 ** attempt * 2
                    print(f"Server error {e.status_code} -- retrying in {wait_time}s (attempt {attempt + 1}/{max_retries})")
                    time.sleep(wait_time)
                    last_error = e
                else:
                    raise
        raise last_error

    def _cache_key(prompt: str, model: str, **kwargs) -> str:
        payload = json.dumps({"prompt": prompt, "model": model, **kwargs}, sort_keys=True)
        return hashlib.sha256(payload.encode()).hexdigest()

    def call_claude(prompt: str, model: str = CLAUDE_MODEL, max_tokens: int = 1024, **kwargs):
        key = _cache_key(prompt, model, max_tokens=max_tokens, **kwargs)
        if key in _response_cache:
            return _response_cache[key], True
        response = _call_with_retry(prompt, model=model, max_tokens=max_tokens, **kwargs)
        text = next(block.text for block in response.content if hasattr(block, "text"))
        _response_cache[key] = text
        return text, False

    PRICING_PER_MILLION_TOKENS = {
        "claude-sonnet-5": {"input": 2.00, "output": 10.00},
        "claude-haiku-4-5-20251001": {"input": 1.00, "output": 5.00},
        "claude-opus-5": {"input": 15.00, "output": 75.00},
    }

    def estimate_cost(num_calls: int, avg_input_tokens: int, avg_output_tokens: int,
                      model: str = CLAUDE_MODEL) -> float:
        if model not in PRICING_PER_MILLION_TOKENS:
            raise ValueError(f"No pricing entry for '{model}'")
        rates = PRICING_PER_MILLION_TOKENS[model]
        input_cost = (num_calls * avg_input_tokens / 1_000_000) * rates["input"]
        output_cost = (num_calls * avg_output_tokens / 1_000_000) * rates["output"]
        total = input_cost + output_cost
        print(f"Estimated cost for {num_calls:,} calls on {model}:")
        print(f"  Input:  {num_calls * avg_input_tokens:,} tokens -> ${input_cost:.2f}")
        print(f"  Output: {num_calls * avg_output_tokens:,} tokens -> ${output_cost:.2f}")
        print(f"  Total:  ${total:.2f}")
        return total

# COMMAND ----------

if "df_filtered" not in globals():
    from pyspark.sql.functions import to_timestamp, date_format, hour, dayofweek, length, when, lower, col

    TABLE_NAME = "miya_academy.default.whatsapp_messages"
    df_expanded = spark.table(TABLE_NAME)
    df_analysis = df_expanded.withColumn(
        "timestamp", to_timestamp(col("DateTimeSent"))
    ).withColumn(
        "date", date_format(col("timestamp"), "yyyy-MM-dd")
    ).withColumn(
        "hour", hour(col("timestamp"))
    ).withColumn(
        "day_of_week", dayofweek(col("timestamp"))
    ).withColumn(
        "message_length", length(col("Text"))
    ).withColumn(
        "sender_type", when(col("From").contains("@"), "Bot").otherwise("User")
    )

    df_filtered = df_analysis.filter(
        ~lower(col("Text")).contains("session has expired") &
        ~lower(col("Text")).contains("session expired") &
        ~lower(col("Text")).contains("session canceled") &
        ~lower(col("Text")).contains("session has been canceled")
    )

# COMMAND ----------

# MAGIC %md
# MAGIC ## Build a stratified sample
# MAGIC
# MAGIC Half the sample comes from conversations that contain a negation ("cancel"/"no"/
# MAGIC "stop"), half from conversations that don't -- so the sample isn't skewed entirely
# MAGIC toward the "interesting" frustrated cases, which would make both classifiers look
# MAGIC more dramatic than the dataset actually is.

# COMMAND ----------

SAMPLE_SIZE_PER_GROUP = 20  # 20 + 20 = 40 conversations total by default. Raise deliberately, not by accident.

negation_convo_ids = df_filtered.filter(
    (col("sender_type") == "User") & (col("Text").rlike("(?i)\\b(cancel|no|nope|stop)\\b"))
).select("ConversationId").distinct()

all_convo_ids = df_filtered.select("ConversationId").distinct()
clean_convo_ids = all_convo_ids.join(negation_convo_ids, on="ConversationId", how="left_anti")

negation_sample_ids = [r["ConversationId"] for r in negation_convo_ids.limit(SAMPLE_SIZE_PER_GROUP).collect()]
clean_sample_ids = [r["ConversationId"] for r in clean_convo_ids.limit(SAMPLE_SIZE_PER_GROUP).collect()]
sample_ids = negation_sample_ids + clean_sample_ids

print(f"Sample: {len(negation_sample_ids)} conversations with a negation, "
      f"{len(clean_sample_ids)} without -- {len(sample_ids)} total")

# COMMAND ----------

# MAGIC %md
# MAGIC ## Pull full transcripts for the sample
# MAGIC
# MAGIC One transcript string per conversation, messages in order, labeled by sender --
# MAGIC this is what gets fed to Claude and to the rule-based classifier.

# COMMAND ----------

sample_messages = df_filtered.filter(col("ConversationId").isin(sample_ids)) \
    .select("ConversationId", "timestamp", "sender_type", "Text") \
    .orderBy("ConversationId", "timestamp") \
    .collect()

transcripts = {}
for row in sample_messages:
    cid = row["ConversationId"]
    transcripts.setdefault(cid, []).append(f"{row['sender_type']}: {row['Text']}")

transcripts = {cid: "\n".join(lines) for cid, lines in transcripts.items()}
print(f"Built {len(transcripts)} transcripts")
print("\nExample transcript:\n" + "-" * 40)
print(next(iter(transcripts.values()))[:500])

# COMMAND ----------

# MAGIC %md
# MAGIC ## Cost estimate -- check this before running the next cell

# COMMAND ----------

avg_transcript_chars = sum(len(t) for t in transcripts.values()) / len(transcripts)
# Rough chars-to-tokens heuristic (~4 chars/token for English); prompt adds ~200 tokens of instructions.
est_input_tokens = int(avg_transcript_chars / 4) + 200
est_output_tokens = 80  # small JSON object

estimate_cost(len(transcripts), est_input_tokens, est_output_tokens)

# COMMAND ----------

# MAGIC %md
# MAGIC ## Claude classification
# MAGIC
# MAGIC One structured call per conversation asking for intent/sentiment/frustration as
# MAGIC JSON -- one call covers all three rather than three separate calls, which is both
# MAGIC cheaper and faster than the "parallel classification" per-attribute approach in the
# MAGIC original architecture sketch.

# COMMAND ----------

import json

CLASSIFY_PROMPT_TEMPLATE = """Classify this WhatsApp customer service conversation. Respond with ONLY a
JSON object, no other text, in exactly this shape:
{{"intent": "<one of: banking, card, loan, insurance, callback, technical_issue, other>",
  "sentiment": "<one of: positive, neutral, negative>",
  "frustration": "<one of: low, medium, high>"}}

Conversation:
{transcript}
"""

def classify_with_claude(transcript: str):
    prompt = CLASSIFY_PROMPT_TEMPLATE.format(transcript=transcript[:3000])  # cap transcript length defensively
    text, was_cached = call_claude(prompt, max_tokens=150)
    try:
        # Claude sometimes wraps JSON in a code fence despite instructions -- strip it if present.
        cleaned = text.strip().strip("`").replace("json\n", "", 1) if text.strip().startswith("```") else text.strip()
        parsed = json.loads(cleaned)
        return parsed, was_cached
    except (json.JSONDecodeError, ValueError):
        return {"intent": "unparseable", "sentiment": "unparseable", "frustration": "unparseable"}, was_cached


ai_results = {}
cache_hits = 0
for i, (cid, transcript) in enumerate(transcripts.items(), 1):
    result, was_cached = classify_with_claude(transcript)
    ai_results[cid] = result
    cache_hits += was_cached
    if i % 10 == 0:
        print(f"Classified {i}/{len(transcripts)}...")

print(f"\nDone. {cache_hits} cache hits out of {len(transcripts)} calls.")

# COMMAND ----------

# MAGIC %md
# MAGIC ## Rule-based baseline
# MAGIC
# MAGIC Same regex-driven approach as the `HybridChatbot` prototype (`04_hybrid_bot_prototype.py`)
# MAGIC -- kept independent here rather than importing that notebook, since it wasn't
# MAGIC structured as a reusable module.

# COMMAND ----------

import re

INTENT_PATTERNS = {
    "banking": r"balance|how much|check account|transfer|statement",
    "card": r"\bcard\b|lost card|stolen|\bpin\b",
    "loan": r"\bloan\b|apply|credit|finance",
    "insurance": r"funeral|claim|policy|beneficiary|cover",
    "callback": r"call me|phone me|speak to|agent|representative",
    "technical_issue": r"not working|error|cant access|wont open|frozen",
}

NEGATION_RE = re.compile(r"\b(cancel|no|nope|stop)\b", re.IGNORECASE)
POSITIVE_RE = re.compile(r"\b(yes|thanks|thank you|great|good|appreciate)\b", re.IGNORECASE)


def classify_with_rules(transcript: str):
    user_lines = [l[len("User: "):] for l in transcript.split("\n") if l.startswith("User: ")]
    full_user_text = " ".join(user_lines).lower()

    intent = "other"
    for name, pattern in INTENT_PATTERNS.items():
        if re.search(pattern, full_user_text):
            intent = name
            break

    negation_count = sum(1 for l in user_lines if NEGATION_RE.search(l))
    positive_count = sum(1 for l in user_lines if POSITIVE_RE.search(l))

    if negation_count >= 2:
        frustration = "high"
    elif negation_count == 1:
        frustration = "medium"
    else:
        frustration = "low"

    if negation_count > positive_count:
        sentiment = "negative"
    elif positive_count > 0:
        sentiment = "positive"
    else:
        sentiment = "neutral"

    return {"intent": intent, "sentiment": sentiment, "frustration": frustration}


rule_results = {cid: classify_with_rules(t) for cid, t in transcripts.items()}

# COMMAND ----------

# MAGIC %md
# MAGIC ## Compare -- agreement rate, not accuracy

# COMMAND ----------

import pandas as pd

comparison_rows = []
for cid in transcripts:
    ai = ai_results[cid]
    rule = rule_results[cid]
    comparison_rows.append({
        "ConversationId": cid,
        "ai_intent": ai["intent"], "rule_intent": rule["intent"], "intent_agree": ai["intent"] == rule["intent"],
        "ai_sentiment": ai["sentiment"], "rule_sentiment": rule["sentiment"], "sentiment_agree": ai["sentiment"] == rule["sentiment"],
        "ai_frustration": ai["frustration"], "rule_frustration": rule["frustration"], "frustration_agree": ai["frustration"] == rule["frustration"],
    })

comparison_pd = pd.DataFrame(comparison_rows)

print("=" * 70)
print("AI vs RULE-BASED -- AGREEMENT RATE (not accuracy -- see note at top)")
print("=" * 70)
print(f"Intent agreement:      {comparison_pd['intent_agree'].mean() * 100:.1f}%")
print(f"Sentiment agreement:   {comparison_pd['sentiment_agree'].mean() * 100:.1f}%")
print(f"Frustration agreement: {comparison_pd['frustration_agree'].mean() * 100:.1f}%")
print(f"\nSample size: {len(comparison_pd)} conversations")

print("\nWhere they disagree on intent (first 10):")
disagreements = comparison_pd[~comparison_pd["intent_agree"]][["ConversationId", "ai_intent", "rule_intent"]]
print(disagreements.head(10).to_string(index=False))

# COMMAND ----------

# MAGIC %md
# MAGIC ## Optional: export a sample for manual ground-truth labeling
# MAGIC
# MAGIC If you want a real accuracy number (not just agreement), label a subset yourself.
# MAGIC This writes a CSV-shaped table with both predictions side by side and a blank
# MAGIC `human_intent` column for you to fill in.

# COMMAND ----------

LABELING_TABLE = "miya_academy.default.classifier_labeling_sample"

labeling_pd = comparison_pd.copy()
labeling_pd["transcript_preview"] = [transcripts[cid][:300] for cid in labeling_pd["ConversationId"]]
labeling_pd["human_intent"] = ""  # fill this in manually, then re-run the comparison against it

spark.createDataFrame(labeling_pd).write.format("delta").mode("overwrite").saveAsTable(LABELING_TABLE)
print(f"Wrote labeling sample to {LABELING_TABLE} ({len(labeling_pd)} rows) -- open it in Catalog Explorer to label by hand.")

# COMMAND ----------

print("=" * 70)
print("03_conversation_classifier complete")
print("=" * 70)
print(f"Sample: {len(transcripts)} conversations | Cache hits: {cache_hits}")
print("Results in: comparison_pd (pandas), ai_results, rule_results (dicts)")
